In [1]:
!pip install -q torch transformers pdfplumber


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.0 MB/s eta 0:00:00


In [2]:
import torch
import pdfplumber
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
MODEL_NAME = "t5-base"

MAX_SOURCE_LEN = 768        # input size
MAX_TARGET_LEN = 250        # output size per page

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("✅ Model loaded on:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Model loaded on: cpu


In [5]:
def extract_pages_from_pdf(pdf_path):
    pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for idx, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if text and text.strip():
                pages.append({
                    "page": idx,
                    "text": text
                })

    return pages

In [6]:
def summarize_page(text, model, tokenizer):
    # --- QUICK WORKAROUND FOR GRAPHS/IMAGES ---
    # If the text is extremely short (likely just labels from a graph),
    # don't force the model to summarize it. Just return it clean.
    word_count = len(text.split())
    if word_count < 20:
        return f"[Graph/Brief Slide - No summary needed] {text.strip()}"

    # --- T5 SUMMARIZATION ---
    inputs = tokenizer(
        "summarize: " + text, # "summarize: " is the standard T5 prompt
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LEN
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            min_length=10,             # 🔥 Lowered so the model doesn't panic on short slides
            num_beams=4,               # 4 is usually the sweet spot for T5
            length_penalty=1.0,
            no_repeat_ngram_size=3,    # 🔥 STOPS LOOPS: Prevents repeating 3-word phrases
            repetition_penalty=1.5,    # 🔥 Penalizes using the same words too often
            early_stopping=True
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [7]:
def summarize_pdf(pdf_path, model, tokenizer):

    pages = extract_pages_from_pdf(pdf_path)

    if not pages:
        return "❌ No readable text found in PDF."

    final_output = []

    for page in pages:
        summary = summarize_page(page["text"], model, tokenizer)

        final_output.append(
            f"📄 Page {page['page']}:\n{summary}\n"
        )

    return "\n".join(final_output)

In [8]:
result = summarize_pdf("test.pdf", model, tokenizer)

print("\n========== PDF PAGE-WISE SUMMARY ==========\n")
print(result)


========== PDF PAGE-WISE SUMMARY ==========

📄 Page 1:
[Graph/Brief Slide - No summary needed] Computer Networks
OSI Model
Dr/ Emad Raouf
1

📄 Page 2:
why OSI model ? - Simplify understanding of data transfer . - Standardization vendors to make all network devices are compatible .

📄 Page 3:
[Graph/Brief Slide - No summary needed] OSI (Open System Interconnection) Model:
• It consists of 7 layers.
Encapsulation
Decapsulation
3

📄 Page 4:
Layer 7: Application Layer: provides services that an application requires . HTTP: used to browse web pages, FTP (file transfer protocol) used to upload and download files . SMTP: (simple mail transfer protocol), used to send mails .

📄 Page 5:
- POP 3 (Post Office Protocol): used to receive emails. - Telnet: Remote login.

📄 Page 6:
Layer 5: Session Layer: - Manages the sessions between applications .

📄 Page 7:
Layer 4: Transport Layer: Functions: 1 - Segmentation of data. 2 - Add port numbers. 3 - Error recovery in case of TCP.

📄 Page 8:
TCP: • Re